## 96.7% is the best accuracy on the validation data we get after performing bayesian optimization.

## 95.1% is the best accuracy on the test set.

In [ ]:
import keras_tuner as kt
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

ModuleNotFoundError: No module named 'keras_tuner'

In [ ]:
dataset, dataset_info = tfds.load("malaria", with_info = True, split = "train", as_supervised = True)

In [ ]:
def split(dataset, train_size, val_size):
    dataset_size = len(dataset)
    train_dataset = dataset.take(int(dataset_size*train_size))
    val_test_dataset = dataset.skip(int(dataset_size*train_size))
    val_dataset = val_test_dataset.take(int(dataset_size*val_size))
    test_dataset = val_test_dataset.skip(int(dataset_size*val_size))
    return train_dataset, val_dataset, test_dataset

In [ ]:
train_dataset, val_dataset, test_dataset = split(dataset, 0.8, 0.1)

In [ ]:
resize_layer = tf.keras.layers.Resizing(256, 256)
rescale_layer = tf.keras.layers.Rescaling(1/255)

def resize_rescale(image, label):
    image = resize_layer(image)
    image = rescale_layer(image)
    return image, label

train_dataset = train_dataset.map(resize_rescale)
val_dataset = val_dataset.map(resize_rescale)
test_dataset = test_dataset.map(resize_rescale)

In [ ]:
test_dataset = test_dataset.batch(1)

In [ ]:
from keras.layers import Normalization, Dense, Input, Conv2D, MaxPool2D, Flatten, BatchNormalization
from keras.losses import BinaryCrossentropy
from keras.optimizers import Adam
from keras.metrics import BinaryAccuracy, TruePositives, TrueNegatives, FalsePositives, FalseNegatives, Precision, Recall, AUC

In [ ]:
class MalariaBayesianTuner(kt.HyperModel):
    def __init__(self, train_dataset, val_dataset):
        super().__init__()
        self.train_dataset = train_dataset
        self.val_dataset = val_dataset

    def build(self, hp):
        # Define hyperparameters search space
        batch_size = hp.Choice('batch_size', values=[16, 24, 32, 48, 64])

        n_conv_layer = hp.Int('n_conv_layer', min_value=1, max_value=5, step=1)
        n_dense_layer = hp.Int('n_dense_layer', min_value=1, max_value=4, step=1)

        conv_filters = {
            1: hp.Choice('conv_filters_1', values=[4, 8, 16]),
            2: hp.Choice('conv_filters_2', values=[16, 32, 64]),
            3: hp.Choice('conv_filters_3', values=[32, 64]),
            4: hp.Choice('conv_filters_4', values=[32, 64]),
            5: hp.Choice('conv_filters_5', values=[32, 64])
        }

        dense_units = {
            1: hp.Choice('dense_units_1', values=[1000, 500, 100]),
            2: hp.Choice('dense_units_2', values=[500, 250, 100]),
            3: hp.Choice('dense_units_3', values=[250, 100, 50]),
            4: hp.Choice('dense_units_4', values=[100, 50, 10]),
        }

        # Create the model
        inputs = Input(shape=(256, 256, 3))
        x = BatchNormalization()(inputs)

        for i in range(1, n_conv_layer + 1):
            x = Conv2D(filters=conv_filters[i],
                      kernel_size=3,
                      strides=1,
                      padding='same',
                      activation="relu",
                      use_bias=False)(x)
            x = BatchNormalization()(x)
            x = MaxPool2D(pool_size=2, strides=2)(x)

        x = Flatten()(x)

        for i in range(1, n_dense_layer + 1):
            x = Dense(
                units=dense_units[i],
                activation='relu'
            )(x)
            x = BatchNormalization()(x)
        x = Dense(1, activation='sigmoid')(x)

        model = tf.keras.Model(inputs=inputs, outputs=x)

        # Compile model
        model.compile(
            optimizer=Adam(),
            loss=BinaryCrossentropy(),
            metrics=[BinaryAccuracy(), TruePositives(), TrueNegatives(),
                    FalsePositives(), FalseNegatives(), Precision(), Recall()]
        )

        return model

    def fit(self, hp, model, *args, **kwargs):
        batch_size = hp.get('batch_size')

        # Prepare datasets with the current batch size
        train_dataset = self.train_dataset.shuffle(buffer_size=500).batch(batch_size)
        val_dataset = self.val_dataset.shuffle(buffer_size=500).batch(batch_size)

        return model.fit(train_dataset, validation_data = val_dataset, **kwargs)

In [ ]:
# Initialize the tuner
tuner = kt.BayesianOptimization(
    hypermodel=MalariaBayesianTuner(train_dataset, val_dataset),
    objective='val_binary_accuracy',
    max_trials=20,
    directory='bayesian_opt',
    project_name='malaria_classification'
)

In [ ]:
learning_rate_callback = tf.keras.callbacks.ReduceLROnPlateau(
  monitor='val_loss',
  factor=0.3,
  patience=2,
  min_lr=1e-6,
  min_delta=0.01,
)

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
            monitor='val_binary_accuracy',           # Metric to monitor
            min_delta=0.001,             # Minimum change to qualify as an improvement
            patience=5,                   # Number of epochs with no improvement after which training will stop
            verbose=1,                    # Print messages about early stopping
            mode='max',                  # The direction is max because we're monitoring loss
            restore_best_weights=True,   # Restore model weights from the epoch with the best value of the monitored quantity
        )

In [ ]:
# Search for best hyperparameters
tuner.search(
    epochs=10,
    callbacks=[
        learning_rate_callback,
        early_stopping
    ]
)

In [ ]:
# Get best hyperparameters and model
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

# Print best hyperparameters
print("Best hyperparameters found:")
print(f"batch Size: {best_hps.get('batch_size')}")
print(f"number of convolution layers: {best_hps.get('n_conv_layer')}")
print(f"number of dense layers: {best_hps.get('n_dense_layer')}")
print(f"Conv Filters 1: {best_hps.get('conv_filters_1')}")
print(f"Conv Filters 2: {best_hps.get('conv_filters_2')}")
print(f"Conv Filters 3: {best_hps.get('conv_filters_3')}")
print(f"Conv Filters 4: {best_hps.get('conv_filters_4')}")
print(f"Conv Filters 5: {best_hps.get('conv_filters_5')}")
print(f"Dense Units 1: {best_hps.get('dense_units_1')}")
print(f"Dense Units 2: {best_hps.get('dense_units_2')}")
print(f"Dense Units 3: {best_hps.get('dense_units_3')}")
print(f"Dense Units 4: {best_hps.get('dense_units_4')}")


In [ ]:
train_dataset = train_dataset.shuffle(buffer_size=500).batch(24)
val_dataset = val_dataset.shuffle(buffer_size=500).batch(24)

In [ ]:
model = tf.keras.Sequential([
    Input(shape = (256, 256, 3)),
    BatchNormalization(),
    Conv2D(filters = 8, kernel_size = 3, strides = 1, padding = 'valid', activation = "relu"),
    BatchNormalization(),
    MaxPool2D(pool_size = 2, strides = 2),
    Conv2D(filters = 32, kernel_size = 3, strides = 1, padding = 'valid', activation = "relu"),
    BatchNormalization(),
    MaxPool2D(pool_size = 2, strides = 2),
    Conv2D(filters = 64, kernel_size = 3, strides = 1, padding = 'valid', activation = "relu"),
    BatchNormalization(),
    MaxPool2D(pool_size = 2, strides = 2),
    Flatten(),
    Dense(100, activation = "relu"),
    BatchNormalization(),
    Dense(100, activation = "relu"),
    BatchNormalization(),
    Dense(1, activation = "sigmoid")
])
metrics = [BinaryAccuracy(), TruePositives(), TrueNegatives(), FalsePositives(), FalseNegatives(), Precision(), Recall()]
model.compile(optimizer = Adam(), loss = BinaryCrossentropy(), metrics = metrics)
history = model.fit(train_dataset, validation_data = val_dataset, epochs = 10, verbose = 1, callbacks = [schedule])

In [ ]:
model.evaluate(test_dataset)